<!-- codex-explainer -->
Check SoX binary/version and dynamic libraries. This isolates environment issues from pipeline logic when augmentation behavior is unexpected.

In [2]:
import subprocess, shlex, os, sys

print("python:", sys.executable)
print(subprocess.check_output(["which", "sox"]).decode().strip())
print(subprocess.check_output(["sox", "--version"]).decode().strip())

# Locate libsox.dylib (try common brew prefixes)
for p in ["/opt/homebrew/lib/libsox.dylib", "/usr/local/lib/libsox.dylib"]:
    print(p, "exists?" , os.path.exists(p))

python: /Users/behkamfallah/Documents/GitHub/indic-language-identification/.venv/bin/python
/opt/homebrew/bin/sox
sox:      SoX v
/opt/homebrew/lib/libsox.dylib exists? True
/usr/local/lib/libsox.dylib exists? False


<!-- codex-explainer -->
Inspect `DYLD_LIBRARY_PATH` to verify where macOS is searching for shared libraries. If SoX-backed torchaudio fails, this variable is often part of the root cause.


In [3]:
import os
print(os.environ.get("DYLD_LIBRARY_PATH"))

/opt/homebrew/lib


<!-- codex-explainer -->
Run a tiny SoX-effect smoke test. If this fails, augmentation paths that depend on SoX effects will fail too, so it is better to detect that up front.


In [4]:
from torchaudio.sox_effects import apply_effects_tensor
import torch

x = torch.zeros(1, 16000)
y, sr = apply_effects_tensor(x, 16000, [["rate", "16000"]])
print("sox OK:", y.shape, sr)

sox OK: torch.Size([1, 16000]) 16000


<!-- codex-explainer -->
Import the core libraries used throughout the pipeline: dataset handling, audio decoding/casting, and feature extraction.


In [5]:
# Imports
import numpy as np
from datasets import Audio, load_dataset
from transformers import AutoFeatureExtractor

/Users/behkamfallah/Documents/GitHub/indic-language-identification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<!-- codex-explainer -->
Define the experiment config in one place. This includes model/data/training choices plus augmentation settings, which makes every later step explicit and reproducible.

Augmentation here means applying controlled perturbations to audio (pitch/spectral/timbre changes) while keeping the language label the same. The goal is to reduce reliance on speaker-specific cues and improve robustness.


In [6]:
config = {
    "run_name": "task2_speaker_obfuscation_mms300m_tuned",
    "output_dir": "./outputs",
    "save_dir": "./models",
    "seed": 42,

    "data": {
        "dataset_id": "badrex/nnti-dataset-full",
        "train_split": "train",
        "eval_split": "validation",
        "audio_column": "audio_filepath",
        "label_column": "language",
        "speaker_column": "speaker_id",
        "sampling_rate": 16000,
        "max_duration_seconds": 8.0,
        "preprocessing_batch_size": 16,
        "preprocessing_keep_in_memory": True,
        "preprocessing_load_from_cache_file": False,
        "preprocessing_writer_batch_size": 16,
    },

    "model": {
        "id": "facebook/mms-300m",
        "apply_dropout": False,
        "dropout": 0.1,
        "freeze_feature_encoder": False,
    },

    "training": {
        "batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "num_train_epochs": 8,
        "learning_rate": 1.0e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "eval_strategy": "steps",
        "eval_steps": 100,
        "save_strategy": "steps",
        "save_steps": 100,
        "logging_steps": 10,
        "save_total_limit": 3,
        "load_best_model_at_end": True,
        "metric_for_best_model": "accuracy",
        "greater_is_better": True,

        # notebook-friendly defaults; flip to True if you are on CUDA
        "fp16": False,
        "bf16": False,

        "gradient_checkpointing": True,
        "dataloader_num_workers": 4,
        "push_to_hub": False,
        "group_by_length": False,
    },

    "augmentation": {
        "enabled": True,
        "train_only": True,
        "apply_prob": 1.0,
        "pitch": {
            "enabled": True,
            "prob": 1.0,
            "cents_min": -280.0,
            "cents_max": 280.0,
        },

        "spectral": {
            "enabled": True,
            "prob": 1.0,
            "notches_min": 1,
            "notches_max": 3,
            "center_freq_min": 250.0,
            "center_freq_max": 3800.0,
            "q_min": 0.8,
            "q_max": 2.2,
            "attenuation_db_min": -18.0,
            "attenuation_db_max": -6.0,
            "highpass_prob": 0.25,
            "highpass_min_hz": 70.0,
            "highpass_max_hz": 250.0,
            "lowpass_prob": 0.25,
            "lowpass_min_hz": 2800.0,
            "lowpass_max_hz": 7000.0,
        },

        "timbre_mask": {
            "enabled": True,
            "prob": 0.8,
            "n_fft": 400,
            "hop_length": 160,
            "win_length": 400,
            "freq_mask_param": 24,
            "time_mask_param": 30,
        },
    },

    "tracking": {
        "use_wandb": True,
        "wandb_project": "Indic-SLID",
        "wandb_run_name": "task2_speaker_obfuscation_mms300m_tuned",
        "hf_token_env": "HF_TOKEN",
        "wandb_api_key_env": "WANDB_API_KEY",
    },
}

<!-- codex-explainer -->
Load train/eval splits and inspect schema/size. This early check catches config mistakes (wrong split names or missing columns) before expensive preprocessing.


In [7]:
# Load raw dataset and inspect columns
dataset_id = config["data"]["dataset_id"]
train_split = config["data"]["train_split"]
eval_split = config["data"]["eval_split"]

ds = load_dataset(dataset_id)
train_ds = ds[train_split].shuffle(seed=config["seed"])
eval_ds = ds[eval_split].shuffle(seed=config["seed"])

print("- Train columns:", train_ds.column_names)
print("- Eval columns:", eval_ds.column_names)
print("- Size of train dataset:", len(train_ds))
print("- Size of eval dataset:", len(eval_ds))

- Train columns: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language']
- Eval columns: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language']
- Size of train dataset: 8689
- Size of eval dataset: 3300


<!-- codex-explainer -->
Inspect a raw row before transformations. Looking at untouched data helps verify assumptions about field names, value formats, and label text.


In [8]:
example_row = train_ds[0]
print("\n- Columns:", example_row.keys())
print("\n- Each column:")
for k in example_row.keys():
    print(f"--> {k}: {example_row[k]}")


- Columns: dict_keys(['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language'])

- Each column:
--> audio_filepath: {'path': 'dfe998f7-3388-4f27-ab8a-b1b20f686675_1_chunk_8.flac', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}
--> duration: 4.097
--> speaker_id: S4256065700339019
--> scenario: Conversation
--> age_group: 18-30
--> job_type: Student
--> area: Rural
--> language: telugu


<!-- codex-explainer -->
Inspect the raw audio payload (`path`, waveform array, sampling rate). This grounds the rest of the pipeline: every model input starts from this decoded signal representation.


In [9]:
print(example_row["audio_filepath"]["path"])
print(example_row["audio_filepath"]["array"], "|", len(example_row["audio_filepath"]["array"]))
print(example_row["audio_filepath"]["sampling_rate"])

dfe998f7-3388-4f27-ab8a-b1b20f686675_1_chunk_8.flac
[0. 0. 0. ... 0. 0. 0.] | 65552
16000


<!-- codex-explainer -->
Load the model’s feature extractor and inspect expected input names.

Why this matters: raw waveform samples are not directly fed to the model. The feature extractor applies model-specific preprocessing (normalization, truncation/padding conventions, attention-mask creation) so inputs match what the pretrained backbone expects.


In [10]:
# Load feature extractor and discover model_input_name
model_id = config["model"]["id"]
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id,
    do_normalize=True,
    return_attention_mask=True
)

model_input_name = feature_extractor.model_input_names[0]
model_input_names = feature_extractor.model_input_names
print("- Model ID:", model_id)
print("- Feature extractor class:", type(feature_extractor))
print("- model_input_name:", model_input_name)
print("- model_input_names:", model_input_names)

- Model ID: facebook/mms-300m
- Feature extractor class: <class 'transformers.models.wav2vec2.feature_extraction_wav2vec2.Wav2Vec2FeatureExtractor'>
- model_input_name: input_values
- model_input_names: ['input_values', 'attention_mask']


<!-- codex-explainer -->
Cast audio to a fixed sampling rate. Standardizing sampling rate is critical because model front-ends assume a consistent temporal scale; mixed sampling rates can silently degrade performance.


In [30]:
# Cast audio column to decoded/resampled audio dicts
audio_col = config["data"]["audio_column"]
sr = int(config["data"]["sampling_rate"])

# Turn the audio columns to the given sampling rate
# All of the dataset uses 16k sampling rate. If we change that in yaml file, this code will change the array of numbers to the given sampling rate.
train_cast = train_ds.cast_column(audio_col, Audio(sampling_rate=sr))

# Take a row of the casted data
example_audio = train_cast[0][audio_col]

print("After casting audio:")
print("- example_audio", example_audio)
print("- audio column keys:", example_audio.keys())
print("- array dtype:", np.asarray(example_audio["array"]).dtype)
print("- array shape:", np.asarray(example_audio["array"]).shape)
print("- sampling_rate:", example_audio["sampling_rate"])

Dataset({
    features: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language'],
    num_rows: 8689
})
After casting audio:
- example_audio {'path': 'dfe998f7-3388-4f27-ab8a-b1b20f686675_1_chunk_8.flac', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}
- audio column keys: dict_keys(['path', 'array', 'sampling_rate'])
- array dtype: float64
- array shape: (65552,)
- sampling_rate: 16000


Let's see if all the sampling rates are 16000.

<!-- codex-explainer -->
Verify sampling-rate consistency across splits. Even if casting is configured, this check confirms that data is in the expected decoded format and uniformly represented.


In [12]:
def sampling_rate_stats(split, name):
    print(f"\n=== Sampling Rate Stats ({name}) ===")
    srs = []
    for i in range(len(split)):  # sample first 500
        audio = split[i][audio_col]
        if isinstance(audio, dict) and "sampling_rate" in audio:
            srs.append(audio["sampling_rate"])
        else:
            print("Audio column not decoded dict format.")
            return

    unique_srs = sorted(set(srs))
    print("Unique sampling rates:", unique_srs)
    print("Counts:")
    for sr in unique_srs:
        print(f"  {sr}: {srs.count(sr)}")

sampling_rate_stats(train_ds, "Train")
sampling_rate_stats(eval_ds, "Eval")


=== Sampling Rate Stats (Train) ===
Unique sampling rates: [16000]
Counts:
  16000: 8689

=== Sampling Rate Stats (Eval) ===
Unique sampling rates: [16000]
Counts:
  16000: 3300


As you can see all of the dataset consistently uses the same sampling rate. So we probably won't need casting.

<!-- codex-explainer -->
Create deterministic `label2id` / `id2label` mappings. Stable class-index ordering is essential for reproducible metrics, checkpoints, and confusion-matrix interpretation.


In [13]:
# Build label mapping (deterministic)
label_col = config["data"]["label_column"]
labels = sorted(train_cast.unique(label_col))
print("- Number of labels:", len(labels))
print("- Labels:", labels)
print("-"*25)

label2id = {lab: i for i, lab in enumerate(labels)}
print("- Label to Id dictionary:", label2id)
print("-"*25)

id2label = {i: lab for lab, i in label2id.items()}
print("- Id to Label dictionary:", id2label)
print("-"*25)

# Check one example label
print("Example label string:", train_cast[0][label_col])
print("Example label id:", label2id[train_cast[0][label_col]])

- Number of labels: 22
- Labels: ['assamese', 'bengali', 'bodo', 'dogri', 'gujarati', 'hindi', 'kannada', 'kashmiri', 'konkani', 'maithili', 'malayalam', 'manipuri', 'marathi', 'nepali', 'odia', 'punjabi', 'sanskrit', 'santali', 'sindhi', 'tamil', 'telugu', 'urdu']
-------------------------
- Label to Id dictionary: {'assamese': 0, 'bengali': 1, 'bodo': 2, 'dogri': 3, 'gujarati': 4, 'hindi': 5, 'kannada': 6, 'kashmiri': 7, 'konkani': 8, 'maithili': 9, 'malayalam': 10, 'manipuri': 11, 'marathi': 12, 'nepali': 13, 'odia': 14, 'punjabi': 15, 'sanskrit': 16, 'santali': 17, 'sindhi': 18, 'tamil': 19, 'telugu': 20, 'urdu': 21}
-------------------------
- Id to Label dictionary: {0: 'assamese', 1: 'bengali', 2: 'bodo', 3: 'dogri', 4: 'gujarati', 5: 'hindi', 6: 'kannada', 7: 'kashmiri', 8: 'konkani', 9: 'maithili', 10: 'malayalam', 11: 'manipuri', 12: 'marathi', 13: 'nepali', 14: 'odia', 15: 'punjabi', 16: 'sanskrit', 17: 'santali', 18: 'sindhi', 19: 'tamil', 20: 'telugu', 21: 'urdu'}
--------

<!-- codex-explainer -->
Instantiate the augmenter from config and print its active subcomponents.

Conceptually, augmentation acts like label-preserving noise: we intentionally vary factors (speaker timbre/pitch/channel-like traits) so the classifier learns language cues that generalize better.


In [14]:
import numpy as np
import torch
from audio_augment_utils import build_speaker_obfuscation_augmenter

augmenter = build_speaker_obfuscation_augmenter(
    config=config,
    sampling_rate=config["data"]["sampling_rate"],
    seed=config["seed"],
)

# Sanity check
print("Augmenter is None?", augmenter is None)
if augmenter is not None:
    # (this is internal but useful for debugging)
    print("Augmenter type:", type(augmenter))
    # Some fields exist only after __post_init__
    print("apply_prob:", getattr(augmenter, "apply_prob", None))
    print("pitch_enabled:", getattr(augmenter, "pitch_enabled", None), "pitch_prob:", getattr(augmenter, "pitch_prob", None))
    print("spectral_enabled:", getattr(augmenter, "spectral_enabled", None), "spectral_prob:", getattr(augmenter, "spectral_prob", None))
    print("timbre_enabled:", getattr(augmenter, "timbre_enabled", None), "timbre_prob:", getattr(augmenter, "timbre_prob", None))


Augmenter is None? False
Augmenter type: <class 'audio_augment_utils.SpeakerObfuscationAugmenter'>
apply_prob: 1.0
pitch_enabled: True pitch_prob: 1.0
spectral_enabled: True spectral_prob: 1.0
timbre_enabled: True timbre_prob: 0.8


<!-- codex-explainer -->
Define preprocessing exactly where augmentation enters the pipeline: waveform -> optional augmentation -> feature extractor -> numeric label encoding.

Placing augmentation before feature extraction is important because we want the model features to reflect augmented acoustics, not post-hoc tensor edits.


In [15]:
def preprocess(examples, apply_augmentation=False, silent_output=False):
    """Exact logic style of your preprocess() in dataset_utils."""

    audio_arrays = [] # We are going to build a list of processed waveforms.

    # Loop over audio samples
    # Each sample is {"path":...,"array":...,"sampling_rate":...}
    for sample in examples[audio_col]:
        audio = np.asarray(sample["array"], dtype=np.float32) # Take the raw waveform and turn it to numpy

        if apply_augmentation and augmenter is not None:
            audio = augmenter(audio)
            print("Applying augmentation.") if not silent_output else None


        audio_arrays.append(audio)

    if not silent_output:
        print(f"- len(audio_arrays):", len(audio_arrays))
        print(f"- len(audio_arrays[0]):", len(audio_arrays[0]))
        print(f"- audio_arrays[0]", audio_arrays[0])
    
    encoded = feature_extractor(
        audio_arrays,
        sampling_rate=sr,
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
    )

    encoded["label"] = [label2id[label] for label in examples[label_col]]
    encoded[model_input_name] = [np.asarray(item) for item in encoded[model_input_name]]
    encoded["length"] = [len(item) for item in encoded[model_input_name]]

    if not silent_output:
        print(f"- type(encoded)", type(encoded))
        print(f"- len(encoded)", len(encoded))
        print(f"- encoded.keys()", encoded.keys())
        print(f"- encoded.values()", encoded.values())

    return encoded, audio_arrays

<!-- codex-explainer -->
Use a tiny batch first for fast debugging. It is much easier to validate keys/shapes and inspect intermediates on one sample than across a full dataset map.


In [16]:
# Take a small batch
batch = train_cast.select(range(1))
print(f"- len(batch):", len(batch))
print(f"- batch:", batch)

print("===============================================")

batch_dict = {k: batch[k] for k in batch.column_names}
print(f"- len(batch_dict):", len(batch_dict))
print(f"- batch_dict.keys():", batch_dict.keys())
print(f"- batch_dict.values():", batch_dict.values())

- len(batch): 1
- batch: Dataset({
    features: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language'],
    num_rows: 1
})
- len(batch_dict): 8
- batch_dict.keys(): dict_keys(['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language'])
- batch_dict.values(): dict_values([[{'path': 'dfe998f7-3388-4f27-ab8a-b1b20f686675_1_chunk_8.flac', 'array': array([0., 0., 0., ..., 0., 0., 0.]), 'sampling_rate': 16000}], [4.097], ['S4256065700339019'], ['Conversation'], ['18-30'], ['Student'], ['Rural'], ['telugu']])


<!-- codex-explainer -->
Compute `max_len` in samples from config duration and sampling rate. This is the concrete truncation boundary used during feature extraction.


In [17]:
sr = int(config["data"]["sampling_rate"])
max_len = int(sr * float(config["data"]["max_duration_seconds"]))

<!-- codex-explainer -->
Run preprocessing without augmentation to establish a clean baseline. This baseline is the reference point for judging whether augmentation changes are meaningful and safe.


In [18]:
# Run without augmentation
encoded_noaug, wav_noaug = preprocess(batch_dict, apply_augmentation=False)

- len(audio_arrays): 1
- len(audio_arrays[0]): 65552
- audio_arrays[0] [0. 0. 0. ... 0. 0. 0.]
- type(encoded) <class 'transformers.feature_extraction_utils.BatchFeature'>
- len(encoded) 4
- encoded.keys() dict_keys(['input_values', 'attention_mask', 'label', 'length'])
- encoded.values() dict_values([[array([7.8985424e-05, 7.8985424e-05, 7.8985424e-05, ..., 7.8985424e-05,
       7.8985424e-05, 7.8985424e-05], dtype=float32)], [array([1, 1, 1, ..., 1, 1, 1], dtype=int32)], [20], [65552]])


<!-- codex-explainer -->
Run the same preprocessing with augmentation enabled. Now we can compare baseline vs augmented outputs under identical conditions.


In [19]:
# Run preprocess WITH augmentation
encoded_aug, wav_aug = preprocess(batch_dict, apply_augmentation=(augmenter is not None))

Applying augmentation.
- len(audio_arrays): 1
- len(audio_arrays[0]): 65552
- audio_arrays[0] [0. 0. 0. ... 0. 0. 0.]
- type(encoded) <class 'transformers.feature_extraction_utils.BatchFeature'>
- len(encoded) 4
- encoded.keys() dict_keys(['input_values', 'attention_mask', 'label', 'length'])
- encoded.values() dict_values([[array([-2.966179e-06, -2.966179e-06, -2.966179e-06, ..., -2.966179e-06,
       -2.966179e-06, -2.966179e-06], dtype=float32)], [array([1, 1, 1, ..., 1, 1, 1], dtype=int32)], [20], [65552]])


<!-- codex-explainer -->
Compare encoded output keys between no-aug and aug paths. Good augmentation should change content, not break the schema expected by Trainer/collator.


In [20]:
print("\n--- Encoded outputs keys (noaug) ---", encoded_noaug.keys())
print("--- Encoded outputs keys (aug)   ---", encoded_aug.keys())


--- Encoded outputs keys (noaug) --- dict_keys(['input_values', 'attention_mask', 'label', 'length'])
--- Encoded outputs keys (aug)   --- dict_keys(['input_values', 'attention_mask', 'label', 'length'])


<!-- codex-explainer -->
Quantify how augmentation changes waveforms (summary stats + normalized difference).

This step answers a practical question: are transforms actually strong enough to diversify data, without completely destroying signal structure?


In [21]:
# Compare waveform-level changes
def waveform_stats(x):
    x = np.asarray(x, dtype=np.float32)
    return {
        "len": int(x.shape[0]),
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "rms": float(np.sqrt(np.mean(x**2))),
    }

print("\n=== Waveform-level comparison (per example) ===")
for i in range(len(wav_noaug)):
    s0 = waveform_stats(wav_noaug[i])
    s1 = waveform_stats(wav_aug[i])
    # Simple similarity metric: normalized L2 difference
    # (0 means identical)
    a = wav_noaug[i]
    b = wav_aug[i]
    # ensure same length before diff (augmentation keeps length, but be safe)
    L = min(len(a), len(b))
    diff = np.linalg.norm(a[:L] - b[:L]) / (np.linalg.norm(a[:L]) + 1e-8)

    print(f"\nExample {i}: label='{batch_dict[label_col][i]}' id={label2id[batch_dict[label_col][i]]}")
    print(" noaug:", s0)
    print(" aug  :", s1)
    print(f" normalized L2 diff: {diff:.4f}")

# Compare feature-extractor outputs (input_values length + attention mask)
print("\n=== Feature-extractor output comparison ===")
for i in range(len(encoded_noaug["label"])):
    len0 = encoded_noaug["length"][i]
    len1 = encoded_aug["length"][i]
    # input sequences are numpy arrays
    x0 = encoded_noaug[model_input_name][i]
    x1 = encoded_aug[model_input_name][i]
    # similarity in feature space
    L = min(len(x0), len(x1))
    feat_diff = np.linalg.norm(x0[:L] - x1[:L]) / (np.linalg.norm(x0[:L]) + 1e-8)
    print(f"Example {i}: input_len noaug={len0}, aug={len1}, feature normalized L2 diff={feat_diff:.4f}")

# Check attention mask integrity
print("\n=== Attention mask sanity ===")
for i in range(len(encoded_noaug["label"])):
    m0 = encoded_noaug["attention_mask"][i]
    m1 = encoded_aug["attention_mask"][i]
    print(f"Example {i}: mask_len noaug={len(m0)} aug={len(m1)} "
          f"| mask sum noaug={int(np.sum(m0))} aug={int(np.sum(m1))}")


=== Waveform-level comparison (per example) ===

Example 0: label='telugu' id=20
 noaug: {'len': 65552, 'mean': -2.432439760013949e-05, 'std': 0.3079604506492615, 'min': -1.0, 'max': 0.999969482421875, 'rms': 0.3079604208469391}
 aug  : {'len': 65552, 'mean': 8.926948567022919e-07, 'std': 0.30095767974853516, 'min': -1.0, 'max': 1.0, 'rms': 0.30095767974853516}
 normalized L2 diff: 1.3958

=== Feature-extractor output comparison ===
Example 0: input_len noaug=65552, aug=65552, feature normalized L2 diff=1.4117

=== Attention mask sanity ===
Example 0: mask_len noaug=65552 aug=65552 | mask sum noaug=65552 aug=65552


<!-- codex-explainer -->
Apply preprocessing via `datasets.map()` to simulate training-time preprocessing at scale while pruning unused columns for efficiency.


In [26]:
# Run datasets.map() and inspect resulting dataset schema
map_bs = int(config["data"]["preprocessing_batch_size"])
print(map_bs)

speaker_col = config["data"]["speaker_column"]
print(speaker_col)

16
speaker_id


In [27]:
keep_cols = [c for c in [speaker_col, label_col] if c in train_cast.column_names]
print(keep_cols)
remove_cols = [c for c in train_cast.column_names if c not in keep_cols]
print(remove_cols)

['speaker_id', 'language']
['audio_filepath', 'duration', 'scenario', 'age_group', 'job_type', 'area']


In [29]:
def preprocess_for_map(batch):
    encoded, _ = preprocess(batch, apply_augmentation=(augmenter is not None), silent_output=True)
    return encoded
    
train_encoded = train_cast.map(
    lambda b: preprocess_for_map(b),
    remove_columns=remove_cols,
    batched=True,
    batch_size=map_bs,
)

print("Encoded dataset columns:", train_encoded.column_names)

Map: 100%|██████████| 8689/8689 [01:26<00:00, 100.15 examples/s]

Encoded dataset columns: ['speaker_id', 'language', 'input_values', 'attention_mask', 'label', 'length']
One encoded example:


In [39]:
print("One encoded example:")

ex = train_encoded[0]
print(len(ex))
print(ex.keys())

One encoded example:
6
dict_keys(['speaker_id', 'language', 'input_values', 'attention_mask', 'label', 'length'])


<!-- codex-explainer -->
Run final sanity checks on encoded data: class-count spread, sequence-length range, and attention-mask alignment. These checks reduce the chance of silent training bugs.


In [23]:
# Sanity checks you should always do
# 1) label distribution
from collections import Counter
cnt = Counter(train_encoded["label"])
print("Label count (min/max):", min(cnt.values()), max(cnt.values()))

# 2) length distribution quick peek
lengths = train_encoded["length"]
print("Length stats: min", min(lengths), "median", np.median(lengths), "max", max(lengths))

# 3) verify attention_mask matches input length for a few examples
for i in range(3):
    xlen = len(train_encoded[i][model_input_name])
    mlen = len(train_encoded[i]["attention_mask"])
    print(f"ex{i}: input_len={xlen}, mask_len={mlen}, match={xlen==mlen}")

Label count (min/max): 337 400
Length stats: min 63985 median 88496.0 max 113760
ex0: input_len=65552, mask_len=65552, match=True
ex1: input_len=85744, mask_len=85744, match=True
ex2: input_len=96256, mask_len=96256, match=True


# TESTS

In [41]:
# Test 1: Speaker overlap + per-language speaker counts (train/eval)
# Assumes:
#   - you already loaded your YAML into `config` (dict)
#   - you have the dataset available on HF Hub (e.g., badrex/nnti-dataset-full)
#
# If you don't have `config` yet, set dataset_id/train_split/eval_split/label_column/speaker_column manually below.

from collections import Counter, defaultdict
from datasets import load_dataset

def test1_speaker_stats(config: dict, max_print_langs: int = 50) -> dict:
    data_cfg = config["data"]
    dataset_id = data_cfg["dataset_id"]
    train_split = data_cfg["train_split"]
    eval_split = data_cfg["eval_split"]
    label_col = data_cfg["label_column"]
    speaker_col = data_cfg["speaker_column"]

    ds = load_dataset(dataset_id)
    train_ds = ds[train_split]
    eval_ds = ds[eval_split]

    # Basic presence checks
    for split_name, split_ds in [("train", train_ds), ("eval", eval_ds)]:
        for col in [label_col, speaker_col]:
            if col not in split_ds.column_names:
                raise KeyError(
                    f"Missing column '{col}' in {split_name} split. "
                    f"Available columns: {split_ds.column_names}"
                )

    # Helper: build mapping language -> set(speakers)
    def lang_to_speakers(dataset):
        m = defaultdict(set)
        for ex in dataset:
            m[ex[label_col]].add(ex[speaker_col])
        return m

    train_map = lang_to_speakers(train_ds)
    eval_map = lang_to_speakers(eval_ds)

    # Overall speaker overlap
    train_speakers = set(train_ds[speaker_col])
    eval_speakers = set(eval_ds[speaker_col])
    inter = train_speakers & eval_speakers

    overall = {
        "train_num_examples": len(train_ds),
        "eval_num_examples": len(eval_ds),
        "train_unique_speakers": len(train_speakers),
        "eval_unique_speakers": len(eval_speakers),
        "speaker_intersection": len(inter),
        "speaker_overlap_ratio_vs_eval": (len(inter) / max(1, len(eval_speakers))),
        "speaker_overlap_ratio_vs_train": (len(inter) / max(1, len(train_speakers))),
    }

    print("=== Test 1: Speaker overlap (overall) ===")
    print(f"Dataset: {dataset_id}")
    print(f"Train split: {train_split} | Eval split: {eval_split}")
    print(f"Train examples: {overall['train_num_examples']} | Eval examples: {overall['eval_num_examples']}")
    print(f"Unique speakers (train): {overall['train_unique_speakers']}")
    print(f"Unique speakers (eval) : {overall['eval_unique_speakers']}")
    print(f"Speakers in both       : {overall['speaker_intersection']}")
    print(f"Overlap vs eval        : {overall['speaker_overlap_ratio_vs_eval']:.3f}")
    print(f"Overlap vs train       : {overall['speaker_overlap_ratio_vs_train']:.3f}")
    print()

    # Per-language stats
    all_langs = sorted(set(train_map.keys()) | set(eval_map.keys()))

    per_lang_rows = []
    for lang in all_langs:
        tr_spk = train_map.get(lang, set())
        ev_spk = eval_map.get(lang, set())
        inter_lang = tr_spk & ev_spk
        per_lang_rows.append(
            {
                "language": lang,
                "train_speakers": len(tr_spk),
                "eval_speakers": len(ev_spk),
                "overlap_speakers": len(inter_lang),
                "overlap_ratio_vs_eval": (len(inter_lang) / max(1, len(ev_spk))),
            }
        )

    # Sort by most overlap first, then by language name
    per_lang_rows.sort(key=lambda r: (-r["overlap_speakers"], r["language"]))

    print("=== Per-language speaker counts & overlap ===")
    print(f"(showing up to {max_print_langs} languages; sorted by overlap desc)")
    for r in per_lang_rows[:max_print_langs]:
        print(
            f"{r['language']:<20} "
            f"train_spk={r['train_speakers']:<2d} "
            f"eval_spk={r['eval_speakers']:<2d} "
            f"overlap={r['overlap_speakers']:<2d} "
            f"overlap_vs_eval={r['overlap_ratio_vs_eval']:.2f}"
        )

    # Quick distribution: how many languages have overlap k
    overlap_hist = Counter(r["overlap_speakers"] for r in per_lang_rows)
    print("\n=== Overlap histogram (languages by #overlap speakers) ===")
    for k in sorted(overlap_hist):
        print(f"{k}: {overlap_hist[k]} languages")

    return {"overall": overall, "per_language": per_lang_rows, "overlap_hist": dict(overlap_hist)}


# Example usage:
results = test1_speaker_stats(config)

# If you DON'T have `config` loaded, you can create a minimal one:
# config = {
#     "data": {
#         "dataset_id": "badrex/nnti-dataset-full",
#         "train_split": "train",
#         "eval_split": "validation",
#         "label_column": "language",
#         "speaker_column": "speaker_id",
#     }
# }
# results = test1_speaker_stats(config)

=== Test 1: Speaker overlap (overall) ===
Dataset: badrex/nnti-dataset-full
Train split: train | Eval split: validation
Train examples: 8689 | Eval examples: 3300
Unique speakers (train): 110
Unique speakers (eval) : 681
Speakers in both       : 0
Overlap vs eval        : 0.000
Overlap vs train       : 0.000

=== Per-language speaker counts & overlap ===
(showing up to 50 languages; sorted by overlap desc)
assamese             train_spk=5  eval_spk=30 overlap=0  overlap_vs_eval=0.00
bengali              train_spk=5  eval_spk=32 overlap=0  overlap_vs_eval=0.00
bodo                 train_spk=5  eval_spk=31 overlap=0  overlap_vs_eval=0.00
dogri                train_spk=5  eval_spk=32 overlap=0  overlap_vs_eval=0.00
gujarati             train_spk=5  eval_spk=30 overlap=0  overlap_vs_eval=0.00
hindi                train_spk=5  eval_spk=32 overlap=0  overlap_vs_eval=0.00
kannada              train_spk=5  eval_spk=33 overlap=0  overlap_vs_eval=0.00
kashmiri             train_spk=5  eval_spk=3